# ARC-05: Two-Body Orbital Dynamics

## Purpose
Model spacecraft translational motion under Newtonian Earth two-body gravity.

## Governing equations
- Position derivative: dr/dt = v.
- Velocity derivative: dv/dt = -(mu / ||r||^3) * r.

## Conventions
- Frame: Earth-Centered Inertial (ECI).
- Position: metres (m).
- Velocity: metres per second (m/s).
- Acceleration: metres per second squared (m/s^2).
- Gravitational parameter: cubic metres per second squared (m^3/s^2).
- State ordering: [x, y, z, vx, vy, vz].

## Scope
Point-mass Earth gravity only. Atmospheric drag, J2 perturbation, thrust, and third-body effects are excluded.


In [1]:
import math
import numpy as np


def two_body_derivative(state_vector, mu):
    if isinstance(state_vector, (str, bytes, bytearray, bool, np.bool_)):
        raise ValueError("State vector must be a numeric array")

    try:
        components = list(state_vector)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State vector must be a numeric array") from exc

    for component in components:
        if isinstance(component, (bool, np.bool_, str, bytes, bytearray, complex, np.complexfloating)):
            raise ValueError("State vector must contain only real numeric values")
        if isinstance(component, (list, tuple, np.ndarray)):
            raise ValueError("State vector must be one-dimensional")

    try:
        untyped_state = np.asarray(state_vector)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State vector must be a numeric array") from exc

    if untyped_state.dtype.kind not in "iuf":
        raise ValueError("State vector must contain only real numeric values")

    try:
        state = np.asarray(untyped_state, dtype=np.float64)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State vector must be float64-compatible") from exc

    if state.shape != (6,):
        raise ValueError("State vector must have exactly six components")

    if not np.all(np.isfinite(state)):
        raise ValueError("State vector must contain only finite values")

    if isinstance(mu, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Gravitational parameter must be numeric")

    try:
        mu = float(mu)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("Gravitational parameter must be numeric") from exc

    if not math.isfinite(mu) or mu <= 0:
        raise ValueError("Gravitational parameter must be finite and positive")

    position = state[:3]
    velocity = state[3:]
    radius = float(np.hypot(np.hypot(abs(position[0]), abs(position[1])), abs(position[2])))

    if not math.isfinite(float(radius)) or radius <= 0:
        raise ValueError("Position radius must be finite and positive")

    position_unit = position / radius
    acceleration_scale = (mu / radius) / radius
    if not math.isfinite(acceleration_scale):
        raise ValueError("Acceleration magnitude is non-finite")
    if acceleration_scale == 0.0:
        raise ValueError("Acceleration magnitude underflows float64")
    acceleration = -acceleration_scale * position_unit
    derivative = np.concatenate((velocity, acceleration))

    if not np.all(np.isfinite(derivative)):
        raise ValueError("Derivative contains non-finite values")

    return derivative


EARTH_MU_M3_S2 = 3.986004418e14
initial_state = np.array([7_000_000.0, 0.0, 0.0, 0.0, 7_500.0, 0.0], dtype=np.float64)
derivative = two_body_derivative(initial_state, EARTH_MU_M3_S2)

print("State derivative:", derivative)


State derivative: [   0.         7500.            0.           -8.13470289   -0.
   -0.        ]


In [2]:
expected_acceleration = -EARTH_MU_M3_S2 / 7_000_000.0**2

assert np.allclose(derivative[:3], initial_state[3:])
assert np.isclose(derivative[3], expected_acceleration)
assert np.allclose(derivative[4:], 0.0)

for invalid_state in (
    [1, 2, 3, 4, 5],
    [1, 2, 3, 4, 5, 6, 7],
    [0, 0, 0, 0, 0, 0],
    [float("nan"), 0, 0, 0, 0, 0],
    ["1", 2, 3, 4, 5, 6],
    [True, 2, 3, 4, 5, 6],
    [1 + 0j, 2, 3, 4, 5, 6],
    [[1], 2, 3, 4, 5, 6],
):
    try:
        two_body_derivative(invalid_state, EARTH_MU_M3_S2)
        raise AssertionError("Invalid state was accepted")
    except ValueError:
        pass

for invalid_mu in (0.0, -1.0, float("nan"), float("inf"), True, "398600441800000"):
    try:
        two_body_derivative(initial_state, invalid_mu)
        raise AssertionError("Invalid gravitational parameter was accepted")
    except ValueError:
        pass

print("Position derivative: PASSED")
print("Gravitational acceleration: PASSED")
print("Invalid state rejection: PASSED")
print("Invalid gravitational parameter rejection: PASSED")


Position derivative: PASSED
Gravitational acceleration: PASSED
Invalid state rejection: PASSED
Invalid gravitational parameter rejection: PASSED
